# ColabDeploy · runtime agent

This notebook turns **this Colab VM** into an inference runtime for your own
control plane. It dials *out* to the gateway over a websocket, so there is no
public IP, no port forwarding and no ngrok.

**What you get**

1. one cell to attach (`launch`)
2. any model you can load in a notebook served behind a stable HTTP endpoint
3. a gateway cache that keeps answering when this session ends

**What you need first**

* a running control plane — see the repo's [README](https://github.com/Nileshsri2022/deploy-ml-models-for-free-colab-ui) for
  the free-tier options, or `python scripts/e2e_local.py` for a local one
* a **runtime token** (`CDP_RT_…`) from that gateway: *UI → Connect a runtime →
  Mint token*

> ⚠️ **Read this before you rely on 24/7 uptime.** Colab ends every session after
> roughly 12 hours, and nothing running inside the VM can prevent that. The agent
> makes re-attaching cheap (artifacts are cached on disk, deployments reconcile
> automatically) and the UI shows a countdown plus a one-click relaunch link, but a
> human still has to re-run the cell. Keeping a free session artificially busy may
> also conflict with Google's usage policy — the browser-side keep-alive is **off by
> default** and should stay off. This project is for learning, prototyping and
> demos; for production, point the same agent at a paid runtime and nothing else
> changes.


In [ ]:
# 1) Install the agent (one time per session)
#
#    Two packages: the shared Pydantic contracts, then the agent itself.
#    Add the extras your model needs - adapters import frameworks lazily, so an
#    uninstalled framework just means that adapter reports itself unavailable.
!pip -q install \
  "git+https://github.com/Nileshsri2022/deploy-ml-models-for-free-colab-ui#subdirectory=packages/shared-schema" \
  "git+https://github.com/Nileshsri2022/deploy-ml-models-for-free-colab-ui#subdirectory=agent"

# Optional, uncomment what you need:
# !pip -q install scikit-learn joblib        # sklearn pickles
# !pip -q install onnxruntime                # .onnx graphs
# !pip -q install xgboost                    # xgboost models
# !pip -q install gdown                      # gdrive:// artifacts
# !pip -q install transformers               # hf:// pipelines
# torch / tensorflow are usually preinstalled on Colab

import colabdeploy

print("colabdeploy", colabdeploy.__version__)

## 2) Point it at your control plane

The gateway URL is fixed. The runtime token is read from **Colab Secrets**
(`CDP_RUNTIME_TOKEN`) so it is not pasted into the notebook.

**Once:** UI → Connect a runtime → Mint token → Colab → 🔑 Secrets → add
`CDP_RUNTIME_TOKEN` = `CDP_RT_…` and grant this notebook access.

**Every session:** Run cells 1–3. Do not mint again unless the token expired,
was rotated, or you lost it.

A runtime token authenticates *this VM*; it is not an API key. Callers of your
endpoints use a separate API key with the `inference` scope.


In [ ]:
import os

GATEWAY_URL = "https://colabdeploy.onrender.com"

def _runtime_token() -> str:
    # 1) Colab Secrets  (Notebook → 🔑 Secrets → CDP_RUNTIME_TOKEN)
    try:
        from google.colab import userdata
        secret = (userdata.get("CDP_RUNTIME_TOKEN") or "").strip()
        if secret:
            return secret
    except Exception:
        pass
    # 2) env (local / Kaggle / CDP_RUNTIME_TOKEN=...)
    return (os.environ.get("CDP_RUNTIME_TOKEN") or "").strip()

TOKEN = _runtime_token()
if not TOKEN:
    raise RuntimeError(
        "No runtime token. In Colab: Secrets → add CDP_RUNTIME_TOKEN = CDP_RT_… "
        "(mint once in the UI). Or set $CDP_RUNTIME_TOKEN."
    )
print("gateway:", GATEWAY_URL)
print("token:  ", TOKEN[:10] + "…" + TOKEN[-4:], f"({len(TOKEN)} chars)")

# Optional overrides
LABEL       = "colab-runtime"           # how this VM appears in the UI
MAX_MODELS  = 4                         # LRU cap on simultaneously warm models (VRAM guard)


In [ ]:
# 3) Attach. Returns immediately - the tunnel runs in a daemon thread, so you can
#    keep training in other cells while this VM serves requests.
from colabdeploy import launch

session = launch(
    gateway_url=GATEWAY_URL,
    token=TOKEN,
    label=LABEL,
    max_loaded_models=MAX_MODELS,
    heartbeat_interval_s=15.0,
    keepalive=True,             # defeats the ~90 min idle timeout
    browser_activity=False,     # leave False: see the ToS note at the top
    warm=True,                  # run each model's warm-up sample after loading
)

In [ ]:
# 4) Check on it at any time (safe to re-run)
import json

import colabdeploy

print(json.dumps(colabdeploy.status(), indent=2, default=str))

## 5) Deploy a model from this notebook

The artifact does **not** have to leave the VM: register it with a `file://` URI
and this machine serves it. Use a Drive or Hugging Face URI instead when you want
*any* runtime to be able to pick the model up.

This needs an **API key with the `manage` scope** (UI → API keys), which is a
different credential from the runtime token above.

Skip this section if you already deployed from the UI — the agent loads whatever
the gateway asks for, automatically.


In [ ]:
MANAGE_KEY = "CDP_API_paste-an-api-key-with-manage-scope"

# --- a) save something to deploy -------------------------------------------
# Replace this with your own model. The example needs no ML framework at all:
# the python_function adapter runs the snippet below as the model.
SNIPPET = '''
POSITIVE = {"good", "great", "love", "amazing", "best"}
NEGATIVE = {"bad", "awful", "hate", "worst", "broken"}

def predict(payload, model=None, meta=None):
    text = str(payload.get("text", ""))
    words = [w.strip(".,!?").lower() for w in text.split()]
    score = sum(w in POSITIVE for w in words) - sum(w in NEGATIVE for w in words)
    return {"label": "positive" if score > 0 else "negative" if score < 0 else "neutral",
            "score": score, "words": len(words)}
'''

# If you have a real model, use this instead:
#   import joblib; joblib.dump(model, "/content/model.joblib")
#   ARTIFACT = {"uri": "file:///content/model.joblib"} ; ADAPTER = "sklearn

In [ ]:
# --- b) register + deploy through the gateway API ---------------------------
import httpx

client = httpx.Client(
    base_url=GATEWAY_URL.rstrip("/"),
    headers={"Authorization": f"Bearer {MANAGE_KEY}"},
    timeout=60.0,
)

model = client.post("/api/models", json={
    "name": "notebook-demo",
    "framework": "python",
    "adapter": "python_function",
    "artifact": {"uri": f"inline:{SNIPPET}", "filename": "model.py"},
    "schema_in": {
        "fields": {"text": {"type": "string", "description": "text to classify"}},
        "required": ["text"],
        "example": {"text": "this is a great little demo"},
    },
    "schema_out": {"fields": {"label": {"type": "string"}, "score": {"type": "integer"}}},
    "warmup_sample": {"inputs": {"text": "amazing, absolutely the best"}},
    "tags": ["notebook"],
}).raise_for_status().json()
print("registered", model["name"], f'v{model["version"]}', model["id"])

deployed = client.post("/api/deploy", json={
    "model_id": model["id"],
    "endpoint": {
        "cache_ttl_s": 300,
        "stale_while_revalidate_s": 3600,
        "degradation_mode": "serve_cached",   # keep answering from cache when this VM dies
        "timeout_ms": 20000,
    },
}).raise_for_status().json()

ROUTE = deployed["endpoint"]["route"]
print(deployed["message"])
print("endpoint:", deployed["endpoint"]["url"])

In [ ]:
# --- c) call your own endpoint from the notebook ----------------------------
response = client.post(
    f"/v1/models/{ROUTE}/predict",
    json={"inputs": {"text": "this is amazing and I love it"}},
).raise_for_status()

print("cache:", response.headers.get("X-ColabDeploy-Cache"))
print("latency:", response.headers.get("X-ColabDeploy-Latency-Ms"), "ms")
print(response.json()["output"])

# Call it again: the second answer comes from the gateway cache, not this VM.
again = client.post(f"/v1/models/{ROUTE}/predict",
                    json={"inputs": {"text": "this is amazing and I love it"}}).json()
print("second call ->", again["cache"], round(again["latency_ms"], 1), "ms")

## 6) When this session ends

Colab will recycle the VM (12 h cap, idle timeout, or a driver update). The
gateway notices within ~3 heartbeat intervals and marks the runtime `stale`, then
`offline`. What happens to your callers depends on each endpoint's
**degradation policy**:

| Policy | Caller sees |
| --- | --- |
| `serve_cached` *(default)* | `200` with the last known answer, `X-ColabDeploy-Cache: STALE`, `"stale": true` |
| `queue` | `202` with a `job_id`; the request runs when a VM returns |
| `fail` | `503` with `Retry-After` |

**To re-arm:** open this notebook (or the
[one-click relaunch link](https://colab.research.google.com/github/Nileshsri2022/deploy-ml-models-for-free-colab-ui/blob/main/agent/colab_agent.ipynb)),
connect a runtime, and re-run cells 1–3. The agent re-registers with the same
token, pulls the artifact from its on-disk cache if the VM was resumed, reconciles
the desired state, and re-warms the top-K request keys the gateway remembered.
Your endpoints and API keys never changed.

Re-running cell 3 is always safe — `launch()` stops the previous session first.


In [ ]:
# Re-attach after a runtime restart / reconnect (idempotent)
session = launch(gateway_url=GATEWAY_URL, token=TOKEN, label=LABEL)

In [ ]:
# Detach cleanly: unload models, close the tunnel, stop keep-alive.
import colabdeploy

colabdeploy.stop()

## Troubleshooting

| Symptom | Cause / fix |
| --- | --- |
| `No runtime token` | add Colab Secret `CDP_RUNTIME_TOKEN` and grant this notebook access |
| `401 UNAUTHORIZED` on launch | token expired or was rotated — mint a new one, update the Secret |
| UI shows `registered` but never `online` | the VM cannot reach the gateway's websocket: check `CDP_PUBLIC_URL` / proxy, and that the host allows `Upgrade` on `/internal/relay` |
| deployment stuck in `loading` | artifact download is slow or failed; check *Runtimes → deployment error* and the log tail |
| `adapter … is not available on this runtime` | the framework isn't installed here — `!pip install scikit-learn` (or set `adapter` explicitly) |
| endpoint answers but always `STALE` | no runtime is online; that is the cache doing its job |
| session died after ~12 h | expected; Colab's hard cap. Re-run cells 1–3 |

### Where things live on this VM

```
/content/.colabdeploy/
├── cache/            artifact blobs + SQLite index (survives a runtime restart)
├── logs/             agent log
├── keepalive         touched every 60s
└── state.json        last known desired state, for a fast resume
```
